# 02 — Data Preprocessing

**Objective:** Clean the CRMLS sold-property data so it is ready for machine learning.  

**Steps we will follow:**
1. Load data and filter to Single Family Residences
2. Select the features (columns) we want to use
3. Handle missing values
4. Convert categorical (text) features to numbers
5. Split into training and testing sets by month
6. Normalize (scale) numerical features
7. Save the cleaned data to CSV files

## 1 — Imports

In [31]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2 — Load & Filter Data

We load all CSV files from the `All Data/` folder and extract the month  
from each row's `CloseDate` so we know which month it came from  
(needed for the chronological train/test split later).

In [32]:
# Resolve project root (parent of Notebooks/) using absolute paths
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..')) if os.path.basename(NOTEBOOK_DIR) == 'Notebooks' else NOTEBOOK_DIR

# Path to the full dataset
DATA_DIR = os.path.join(PROJECT_ROOT, 'All Data')

# Output directory for train.csv, test.csv, cleaned_data.csv
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'Cleaned, Test, Train Data')

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:     {DATA_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')

# Find all CRMLSSold CSV files
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, 'CRMLSSold*.csv')))
print(f'Found {len(csv_files)} CSV files')

# Load each file and extract month from CloseDate
dfs = []
for f in csv_files:
    temp = pd.read_csv(f, low_memory=False)
    
    # Extract YYYYMM from CloseDate (works for all files including multi-year ones)
    close_dt = pd.to_datetime(temp['CloseDate'], errors='coerce')
    temp['month'] = close_dt.dt.strftime('%Y%m')
    
    dfs.append(temp)
    print(f'  Loaded {os.path.basename(f)}: {len(temp):,} rows')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'\nTotal: {len(csv_files)} files — {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')

Project root: c:\Users\desai\Desktop\IDX Data Science Internship\Code
Data dir:     c:\Users\desai\Desktop\IDX Data Science Internship\Code\All Data
Output dir:   c:\Users\desai\Desktop\IDX Data Science Internship\Code\Cleaned, Test, Train Data
Found 31 CSV files
  Loaded CRMLSSold20220101_20231231_filled.csv: 157,828 rows
  Loaded CRMLSSold202401_filled.csv: 17,958 rows
  Loaded CRMLSSold202402_filled.csv: 19,925 rows
  Loaded CRMLSSold202403_filled.csv: 23,276 rows
  Loaded CRMLSSold202404_filled.csv: 24,640 rows
  Loaded CRMLSSold202405_filled.csv: 26,487 rows
  Loaded CRMLSSold202406_filled.csv: 24,328 rows
  Loaded CRMLSSold202407_filled.csv: 26,240 rows
  Loaded CRMLSSold202408.csv: 24,558 rows
  Loaded CRMLSSold202409.csv: 21,267 rows
  Loaded CRMLSSold202410.csv: 23,274 rows
  Loaded CRMLSSold202411.csv: 20,279 rows
  Loaded CRMLSSold202412.csv: 20,241 rows
  Loaded CRMLSSold202501_filled.csv: 18,738 rows
  Loaded CRMLSSold202502.csv: 18,702 rows
  Loaded CRMLSSold202503.csv: 2

In [33]:
# Filter to Residential / Single Family Residence only and remove extreme MLS data entry errors / outliers
df = df_raw[
    (df_raw['PropertyType'] == 'Residential') &
    (df_raw['PropertySubType'] == 'SingleFamilyResidence') &
    (df_raw['ClosePrice'] >= 100000) & (df_raw['ClosePrice'] <= 5000000) &
    (df_raw['LivingArea'] >= 300) & (df_raw['LivingArea'] <= 10000) &
    (df_raw['BathroomsTotalInteger'] >= 1) & (df_raw['BathroomsTotalInteger'] <= 10) &
    (df_raw['BedroomsTotal'] >= 1) & (df_raw['BedroomsTotal'] <= 10) &
    (df_raw['month'].notnull())
].copy()

months_found = sorted(df['month'].unique())
print(f'After filtering: {df.shape[0]:,} rows  ({df.shape[0] / df_raw.shape[0] * 100:.1f}% of raw data)')
print(f'Months spanned: {months_found[0]} to {months_found[-1]} ({len(months_found)} total months)')

After filtering: 404,273 rows  (49.4% of raw data)
Months spanned: 202201 to 202606 (54 total months)


## 3 — Select Features

Out of the 82 columns, we pick 10 features that are most likely to predict `ClosePrice`.  
We keep the selection small and simple for now — more features can be added in Week 6.

In [34]:
# Features we will use to predict ClosePrice
features = [
    'LivingArea',            # Square footage of living area
    'BedroomsTotal',         # Number of bedrooms
    'BathroomsTotalInteger', # Number of bathrooms
    'LotSizeSquareFeet',     # Lot size in square feet
    'YearBuilt',             # Year the property was built
    'GarageSpaces',          # Number of garage spaces
    'Stories',               # Number of stories
    'DaysOnMarket',          # Days on market before sale
    'City',                  # City name (text — will be encoded later)
    'PostalCode',            # ZIP code  (text — will be encoded later)
]

# The value we want to predict
target = 'ClosePrice'

# Keep only the columns we need + target + month (for splitting)
keep_cols = features + [target, 'month']
df = df[keep_cols].copy()

print(f'Selected {len(features)} features + target + month column')
print(f'Shape: {df.shape}')
df.head()

Selected 10 features + target + month column
Shape: (404273, 12)


,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,Stories,DaysOnMarket,City,PostalCode,ClosePrice,month
3,2645.0,4.0,4.0,13376.0,2016.0,2.0,NaN,37.0,Carlsbad,92008,2499999.0,202201
6,2070.0,3.0,3.0,3397.0,2007.0,1.0,NaN,13.0,Lake Arrowhead,92352,640000.0,202201
7,1174.0,3.0,2.0,9900.0,1960.0,2.0,1.0,3.0,Fontana,92336,438000.0,202203
12,1996.0,2.0,2.0,6098.0,2005.0,2.0,NaN,81.0,Cathedral City,92234,615000.0,202201
14,1422.0,3.0,2.0,12197.0,2021.0,2.0,1.0,63.0,Paradise,95969,399990.0,202201


## 4 — Handle Missing Values

**Strategy:**
- First, drop any rows where `ClosePrice` (our target) is missing — we can't train on those.
- For **numeric** columns: fill missing values with the **median** (the middle value).  
  Median is better than mean because it isn't affected by extreme outliers.
- For **categorical** (text) columns: fill missing values with `"Unknown"`.

In [35]:
# Let's see how many missing values each column has
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df):,}')

Missing values per column:
LivingArea                   0
BedroomsTotal                0
BathroomsTotalInteger        0
LotSizeSquareFeet         6915
YearBuilt                  189
GarageSpaces             13872
Stories                  53055
DaysOnMarket                 0
City                       512
PostalCode                   3
ClosePrice                   0
month                        0
dtype: int64

Total rows: 404,273


In [36]:
# Step 1: Drop rows where ClosePrice is missing (we need it to train the model)
before = len(df)
df = df.dropna(subset=[target])
print(f'Dropped {before - len(df)} rows with missing ClosePrice')

# Step 2: Define numeric vs categorical columns
numeric_features = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
    'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces',
    'Stories', 'DaysOnMarket'
]
categorical_features = ['City', 'PostalCode']

# Step 3: Fill missing numeric values with the median
for col in numeric_features:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'  Filled {col} missing values with median = {median_val}')

# Step 4: Fill missing categorical values with "Unknown"
for col in categorical_features:
    df[col] = df[col].fillna('Unknown')
    print(f'  Filled {col} missing values with "Unknown"')

# Verify: no more missing values
print(f'\nRemaining missing values: {df.isnull().sum().sum()}')

Dropped 0 rows with missing ClosePrice
  Filled LivingArea missing values with median = 1789.0
  Filled BedroomsTotal missing values with median = 3.0
  Filled BathroomsTotalInteger missing values with median = 2.0
  Filled LotSizeSquareFeet missing values with median = 7206.0
  Filled YearBuilt missing values with median = 1975.0
  Filled GarageSpaces missing values with median = 2.0
  Filled Stories missing values with median = 1.0
  Filled DaysOnMarket missing values with median = 16.0
  Filled City missing values with "Unknown"
  Filled PostalCode missing values with "Unknown"

Remaining missing values: 0


## 5 — Encode Categorical Features

Machine learning models need **numbers**, not text.  
We use `LabelEncoder` to assign a unique number to each city and postal code.

**Note:** Label encoding assigns arbitrary numbers (e.g., "Los Angeles" → 5, "San Diego" → 8).  
This works well for tree-based models (Decision Trees, Random Forests, etc.)  
but linear models might interpret these numbers as having an order.  
We'll keep it simple for now.

In [37]:
# Make sure PostalCode is a string before encoding
# (it might have been loaded as a number like 94401.0)
df['PostalCode'] = df['PostalCode'].astype(str)
df['PostalCode'] = df['PostalCode'].str.replace(r'\.0$', '', regex=True)

# Create label encoders
city_encoder = LabelEncoder()
postal_encoder = LabelEncoder()

# Fit and transform — converts text to numbers
df['City'] = city_encoder.fit_transform(df['City'])
df['PostalCode'] = postal_encoder.fit_transform(df['PostalCode'])

print(f'City: {len(city_encoder.classes_)} unique values encoded')
print(f'PostalCode: {len(postal_encoder.classes_)} unique values encoded')
print('\nFirst 5 rows after encoding:')
df.head()

City: 1152 unique values encoded
PostalCode: 3725 unique values encoded

First 5 rows after encoding:


,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,Stories,DaysOnMarket,City,PostalCode,ClosePrice,month
3,2645.0,4.0,4.0,13376.0,2016.0,2.0,1.0,37.0,158,384,2499999.0,202201
6,2070.0,3.0,3.0,3397.0,2007.0,1.0,1.0,13.0,511,535,640000.0,202201
7,1174.0,3.0,2.0,9900.0,1960.0,2.0,1.0,3.0,339,525,438000.0,202203
12,1996.0,2.0,2.0,6098.0,2005.0,2.0,1.0,81.0,172,471,615000.0,202201
14,1422.0,3.0,2.0,12197.0,2021.0,2.0,1.0,63.0,765,3642,399990.0,202201


## 6 — Train/Test Split by Month

We use a **chronological split**:  
- **Training set** = all months except the last `TEST_MONTHS`  
- **Test set** = the last `TEST_MONTHS` months  

Change `TEST_MONTHS` below to adjust the holdout window.

In [38]:
# ============================================================
# TUNABLE PARAMETER
# Change this number to hold out more or fewer test months.
# ============================================================
TEST_MONTHS = 2

# All available months in our data (sorted chronologically)
all_months = sorted(df['month'].unique())
print(f'Available months ({len(all_months)}): {all_months[0]} to {all_months[-1]}')

# Training = everything except the last TEST_MONTHS
train_months = all_months[:-TEST_MONTHS]
# Test = the last TEST_MONTHS months
test_months  = all_months[-TEST_MONTHS:]

print(f'\nTraining months: {train_months[0]} to {train_months[-1]} ({len(train_months)} months)')
print(f'Test months:     {test_months[0]} to {test_months[-1]} ({len(test_months)} months)')

# Split the data
train_df = df[df['month'].isin(train_months)].copy()
test_df  = df[df['month'].isin(test_months)].copy()

# Drop the 'month' column — it was only needed for splitting, not for prediction
train_df = train_df.drop(columns=['month'])
test_df  = test_df.drop(columns=['month'])

total = len(train_df) + len(test_df)
print(f'\nTraining set: {len(train_df):,} rows ({len(train_df)/total*100:.1f}%)')
print(f'Test set:     {len(test_df):,} rows ({len(test_df)/total*100:.1f}%)')

Available months (54): 202201 to 202606

Training months: 202201 to 202604 (52 months)
Test months:     202605 to 202606 (2 months)

Training set: 379,894 rows (94.0%)
Test set:     24,379 rows (6.0%)


## 7 — Normalize Numerical Features

`StandardScaler` transforms each numeric feature so it has **mean ≈ 0** and **std ≈ 1**.  
This helps some models (like Linear Regression) work better.

**Important:** We fit the scaler on the **training data only**, then apply (transform) it to  
both training and testing data. This prevents **data leakage** — the model should not  
learn anything from the test data during training.

In [39]:
scaler = StandardScaler()

# Fit on training data AND transform it
train_df[numeric_features] = scaler.fit_transform(train_df[numeric_features])

# Transform test data using the SAME scaler (do NOT fit again!)
test_df[numeric_features] = scaler.transform(test_df[numeric_features])

print('Training set stats after scaling (should be mean ≈ 0, std ≈ 1):')
print(train_df[numeric_features].describe().round(2))

Training set stats after scaling (should be mean ≈ 0, std ≈ 1):
       LivingArea  BedroomsTotal  BathroomsTotalInteger  LotSizeSquareFeet  \
count   379894.00      379894.00              379894.00          379894.00   
mean         0.00          -0.00                   0.00               0.00   
std          1.00           1.00                   1.00               1.00   
min         -1.92          -2.66                  -1.55              -0.02   
25%         -0.70          -0.49                  -0.55              -0.02   
50%         -0.22          -0.49                  -0.55              -0.02   
75%          0.47           0.59                   0.44              -0.02   
max          9.22           7.10                   7.41             156.18   

       YearBuilt  GarageSpaces    Stories  DaysOnMarket  
count  379894.00     379894.00  379894.00     379894.00  
mean        0.00          0.00      -0.00         -0.00  
std         1.00          1.00       1.00          1.00  
m

## 8 — Save Cleaned Data

We save three files:
- `train.csv` — training data (scaled)
- `test.csv` — test data (scaled)
- `cleaned_data.csv` — all data, encoded but NOT scaled (useful for exploration later)

In [40]:
# Save the scaled train and test sets
train_df.to_csv(os.path.join(OUTPUT_DIR, 'train.csv'), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, 'test.csv'), index=False)

# Save the full cleaned dataset (encoded but NOT scaled)
# 'df' still has the original numeric values with encoded categories
df_clean = df.drop(columns=['month'])
df_clean.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_data.csv'), index=False)

print(f'Saved files to {OUTPUT_DIR}:')
print(f'  train.csv        — {len(train_df):,} rows (scaled)')
print(f'  test.csv         — {len(test_df):,} rows (scaled)')
print(f'  cleaned_data.csv — {len(df_clean):,} rows (encoded, not scaled)')

Saved files to c:\Users\desai\Desktop\IDX Data Science Internship\Code\Cleaned, Test, Train Data:
  train.csv        — 379,894 rows (scaled)
  test.csv         — 24,379 rows (scaled)
  cleaned_data.csv — 404,273 rows (encoded, not scaled)


## 9 — Summary

### What we did:
1. **Loaded** the full multi-year CRMLS dataset (2022–2026) and filtered to Single Family Residences
2. **Selected** 10 features: LivingArea, Bedrooms, Bathrooms, LotSize, YearBuilt, GarageSpaces, Stories, DaysOnMarket, City, PostalCode
3. **Handled missing values**: median for numeric columns, "Unknown" for categorical columns
4. **Encoded** City and PostalCode from text to numbers using LabelEncoder
5. **Split** data chronologically: last 2 months = test, all preceding months = training
6. **Normalized** numeric features using StandardScaler (fit on training data only)
7. **Saved** cleaned data to CSV files

### What's next (Week 4):
Train a **Linear Regression** baseline model using this cleaned data.